In [1]:
from tensorflow.keras.datasets import cifar100
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models

(x_train, y_train), (x_test, y_test) = cifar100.load_data()
x_train, x_test = x_train/255.0, x_test/255.0

y_train = to_categorical(y_train, 100)
y_test = to_categorical(y_test, 100)

169001437/169001437 ━━━━━━━━━━━━━━━━━━━━ 15s 0us/step


In [16]:
x_train.shape

(50000, 32, 32, 3)

In [2]:
from keras.models import Sequential
from keras.layers import Conv2D, AveragePooling2D, Flatten, Dense

model = Sequential()
model.add(Conv2D(filters=6, kernel_size=5, strides=1, activation='tanh',
                 input_shape=(32,32,3), padding='same'))
model.add(AveragePooling2D(pool_size=2, strides=2, padding='valid'))
model.add(Conv2D(filters=16, kernel_size=5, strides=1, activation='tanh',
                 padding='valid'))
model.add(AveragePooling2D(pool_size=2, strides=2, padding='valid'))
model.add(Conv2D(filters=120, kernel_size=5, strides=1, activation='tanh',
                 padding='valid'))
model.add(Flatten())

model.add(Dense(units=200, activation='tanh'))
model.add(Dense(units=100, activation='softmax'))
model.summary()

e:\Users\PERSONAL\miniconda3\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 6)      │           456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 16, 16, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 12, 12, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 6, 6, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 2, 2, 120)      │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 480)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 200)            │        96,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        20,100 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 167,292 (653.48 KB)

 Trainable params: 167,292 (653.48 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
def lr_schedule(epoch):
    if epoch <= 2:
        lr = 5e-4
    elif epoch > 2 and epoch <= 5:
        lr = 2e-4
    elif epoch > 5 and epoch <= 9:
        lr = 5e-5
    else:
        lr = 1e-5
    return lr

In [ ]:
from keras.callbacks import ModelCheckpoint, LearningRateScheduler

lr_reducer = LearningRateScheduler(lr_schedule)
checkpoint = ModelCheckpoint(filepath='../models/LetNet5.keras',
                             monitor='val_accuracy',
                             verbose=1,
                             save_best_only=True,
                             mode='max')
callbacks = [checkpoint, lr_reducer]
model.compile(loss='categorical_crossentropy', optimizer='sgd',
              metrics=['accuracy'])
hist = model.fit(x_train, y_train, batch_size=32, epochs=20,
                 validation_data=(x_test, y_test), callbacks=callbacks,
                 verbose=2, shuffle=True)

Epoch 1/20

Epoch 1: val_accuracy improved from None to 0.01690, saving model to ../models/LetNet5.keras
1563/1563 - 7s - 4ms/step - accuracy: 0.0149 - loss: 4.5962 - val_accuracy: 0.0169 - val_loss: 4.5924 - learning_rate: 5.0000e-04
Epoch 2/20

Epoch 2: val_accuracy improved from 0.01690 to 0.02060, saving model to ../models/LetNet5.keras
1563/1563 - 6s - 4ms/step - accuracy: 0.0211 - loss: 4.5871 - val_accuracy: 0.0206 - val_loss: 4.5813 - learning_rate: 5.0000e-04
Epoch 3/20

Epoch 3: val_accuracy improved from 0.02060 to 0.02500, saving model to ../models/LetNet5.keras
1563/1563 - 6s - 4ms/step - accuracy: 0.0251 - loss: 4.5720 - val_accuracy: 0.0250 - val_loss: 4.5609 - learning_rate: 5.0000e-04
Epoch 4/20

Epoch 4: val_accuracy improved from 0.02500 to 0.02570, saving model to ../models/LetNet5.keras
1563/1563 - 7s - 4ms/step - accuracy: 0.0277 - loss: 4.5544 - val_accuracy: 0.0257 - val_loss: 4.5481 - learning_rate: 2.0000e-04
Epoch 5/20

Epoch 5: val_accuracy did not improve f

In [3]:
from keras.models import Sequential
from keras.regularizers import l2
from keras.layers import Conv2D, AveragePooling2D, Flatten, Dense, Activation, MaxPool2D, BatchNormalization, Dropout

model = Sequential()
model.add(Conv2D(filters=96, kernel_size=(11,11), strides=(2,2), padding='valid',
                 input_shape=(32,32,3)))
model.add(Activation('relu'))
model.add(MaxPool2D(pool_size=(3,3), strides=(2,2)))
model.add(BatchNormalization())

model.add(Conv2D(filters=256, kernel_size=(5,5), strides=(1,1), padding='same',
                 kernel_regularizer=l2(0.0005)))
model.add(Activation('relu'))
model.add(MaxPool2D(pool_size=(3,3), strides=(1,1), padding='valid'))
model.add(BatchNormalization())

model.add(Conv2D(filters=384, kernel_size=(3,3), strides=(1,1), padding='same',
                 kernel_regularizer=l2(0.0005)))
model.add(Activation('relu'))
model.add(BatchNormalization())

model.add(Conv2D(filters=256, kernel_size=(3,3), strides=(1,1), padding='same',
                 kernel_regularizer=l2(0.0005)))
model.add(Activation('relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(3,3), strides=(1,1), padding='valid'))

model.add(Flatten())

model.add(Dense(units=4096, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(units=4096, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(units=100, activation='softmax'))
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 11, 11, 96)     │        34,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 11, 11, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 5, 5, 96)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 5, 5, 96)       │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 5, 5, 256)      │       614,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 5, 5, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 3, 3, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 3, 3, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 3, 3, 384)      │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 3, 3, 384)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 3, 3, 384)      │         1,536 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 3, 3, 256)      │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 3, 3, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 3, 3, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 1, 1, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4096)           │     1,052,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 100)            │       409,700 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,667,364 (78.84 MB)

 Trainable params: 20,665,380 (78.83 MB)

 Non-trainable params: 1,984 (7.75 KB)

In [4]:
from tensorflow import keras
import numpy as np

reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=np.sqrt(0.1))
optimizer = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)

model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy']) 
model.fit(x_train, y_train, batch_size=128, epochs=20, validation_data=(x_test, y_test), verbose=2, callbacks=[reduce_lr])

Epoch 1/20
391/391 - 82s - 211ms/step - accuracy: 0.0934 - loss: 4.3404 - val_accuracy: 0.0918 - val_loss: 4.5338 - learning_rate: 0.0100
Epoch 2/20
391/391 - 78s - 200ms/step - accuracy: 0.1838 - loss: 3.7224 - val_accuracy: 0.1293 - val_loss: 4.1723 - learning_rate: 0.0100
Epoch 3/20
391/391 - 83s - 211ms/step - accuracy: 0.2447 - loss: 3.3720 - val_accuracy: 0.2371 - val_loss: 3.4463 - learning_rate: 0.0100
Epoch 4/20
391/391 - 83s - 212ms/step - accuracy: 0.2904 - loss: 3.1168 - val_accuracy: 0.2346 - val_loss: 3.5004 - learning_rate: 0.0100
Epoch 5/20
391/391 - 80s - 203ms/step - accuracy: 0.3316 - loss: 2.9132 - val_accuracy: 0.3107 - val_loss: 3.0255 - learning_rate: 0.0100
Epoch 6/20
391/391 - 80s - 204ms/step - accuracy: 0.3688 - loss: 2.7441 - val_accuracy: 0.3450 - val_loss: 2.9210 - learning_rate: 0.0100
Epoch 7/20
391/391 - 79s - 202ms/step - accuracy: 0.3980 - loss: 2.6008 - val_accuracy: 0.3252 - val_loss: 3.0321 - learning_rate: 0.0100
Epoch 8/20
391/391 - 79s - 201ms/s

In [ ]:
model = Sequential()

#Block1
model.add(Conv2D(filters=64, kernel_size=(3,3), strides=(1,1), activation='relu', padding='same', input_shape=(32,32,3)))
model.add(Conv2D(filters=64, kernel_size=(3,3), strides=(1,1), activation='relu', padding='same'))
model.add(MaxPool2D((2,2), strides=(2,2)))
#Block2
model.add(Conv2D(filters=128, kernel_size=(3,3), strides=(1,1), activation='relu', padding='same'))
model.add(Conv2D(filters=128, kernel_size=(3,3), strides=(1,1), activation='relu', padding='same'))
model.add(MaxPool2D((2,2), strides=(2,2)))
#Block3
model.add(Conv2D(filters=256, kernel_size=(3,3), strides=(1,1), activation='relu', padding='same'))
model.add(Conv2D(filters=256, kernel_size=(3,3), strides=(1,1), activation='relu', padding='same'))
model.add(Conv2D(filters=256, kernel_size=(3,3), strides=(1,1), activation='relu', padding='same'))
model.add(MaxPool2D((2,2), strides=(2,2)))
#Block4
model.add(Conv2D(filters=512, kernel_size=(3,3), strides=(1,1), activation='relu', padding='same'))
model.add(Conv2D(filters=512, kernel_size=(3,3), strides=(1,1), activation='relu', padding='same'))
model.add(Conv2D(filters=512, kernel_size=(3,3), strides=(1,1), activation='relu', padding='same'))
model.add(MaxPool2D((2,2), strides=(2,2)))
#Block5
model.add(Conv2D(filters=512, kernel_size=(3,3), strides=(1,1), activation='relu', padding='same'))
model.add(Conv2D(filters=512, kernel_size=(3,3), strides=(1,1), activation='relu', padding='same'))
model.add(Conv2D(filters=512, kernel_size=(3,3), strides=(1,1), activation='relu', padding='same'))
model.add(MaxPool2D((2,2), strides=(2,2)))
#Block5
model.add(Flatten())
model.add(Dense(1024, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1024, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(100, activation='softmax'))
model.summary()

In [ ]:
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=np.sqrt(0.1))
optimizer = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)

model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy']) 
model.fit(x_train, y_train, batch_size=128, epochs=20, validation_data=(x_test, y_test), verbose=2, callbacks=[reduce_lr])

In [5]:
from tensorflow.keras.layers import concatenate
from tensorflow.keras.initializers import HeNormal, Zeros

def inception_module(x, filters_1x1, filters_3x3_reduce, filters_3x3,
                     filters_5x5_reduce, filters_5x5, filters_pool_proj, name=None):
    
    kernel_init = keras.initializers.glorot_uniform()
    bias_init = keras.initializers.Constant(value=0.2)

    conv_1x1 = Conv2D(filters_1x1, kernel_size=(1,1), padding='same',
                    activation='relu', kernel_initializer=kernel_init, bias_initializer=bias_init)(x)

    pre_cov_3x3 = Conv2D(filters_3x3_reduce, kernel_size=(1,1), padding='same',
                         activation='relu', kernel_initializer=kernel_init, bias_initializer=bias_init)(x)
    
    conv_3x3 = Conv2D(filters_3x3, kernel_size=(3,3), padding='same', activation='relu',
                      kernel_initializer=kernel_init, bias_initializer=bias_init)(pre_cov_3x3)
    
    pre_conv_5x5 = Conv2D(filters_5x5_reduce, kernel_size=(1,1), padding='same', activation='relu',
                          kernel_initializer=kernel_init, bias_initializer=bias_init)(x)
    
    conv_5x5 = Conv2D(filters_5x5, kernel_size=(5,5), padding='same', activation='relu',
                      kernel_initializer=kernel_init, bias_initializer=bias_init)(pre_conv_5x5)
    
    pool_proj = MaxPool2D((3,3), strides=(1,1), padding='same')(x)
    pool_proj = Conv2D(filters_pool_proj, (1,1), padding='same', activation='relu',
                       kernel_initializer=kernel_init, bias_initializer=bias_init)(pool_proj)

    output = concatenate([conv_1x1, conv_3x3, conv_5x5, pool_proj], axis=3, name=name)
    return output

In [ ]:
input = layers.Input(shape=(32,32,3))

kernel_init = keras.initializers.glorot_uniform()
bias_init = keras.initializers.Constant(value=0.2)

x = Conv2D(64, (7,7), padding='same', strides=(2,2), activation='relu',
           name='conv_1_7x7_2', kernel_initializer=kernel_init, bias_initializer=bias_init)(input)
x = MaxPool2D((3,3), padding='same', strides=(2,2), name='max_pool_1_3x3_2')(x)
x = BatchNormalization()(x)
x = Conv2D(64, (1,1), padding='same', strides=(1,1), activation='relu')(x)
x = Conv2D(192, (3,3), padding='same', strides=(1,1), activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPool2D((3,3), padding='same', strides=(2,2))(x)

x = inception_module(x, filters_1x1=64, filters_3x3_reduce=96, filters_3x3=128,
                     filters_5x5_reduce=16, filters_5x5=32, filters_pool_proj=32,
                     name='inception_3a')
x = inception_module(x, filters_1x1=128, filters_3x3_reduce=128, filters_3x3=192,
                     filters_5x5_reduce=32, filters_5x5=96, filters_pool_proj=64,
                     name='inception_3b')
x = MaxPool2D((3,3), padding='same', strides=(2,2))(x)

x = inception_module(x, filters_1x1=192, filters_3x3_reduce=96, filters_3x3=208,
                     filters_5x5_reduce=16, filters_5x5=48, filters_pool_proj=64,
                     name='inception_4a')

x = inception_module(x, filters_1x1=160, filters_3x3_reduce=112, filters_3x3=224,
                     filters_5x5_reduce=24, filters_5x5=64, filters_pool_proj=64,
                     name='inception_4b')

x = inception_module(x, filters_1x1=128, filters_3x3_reduce=128, filters_3x3=256,
                     filters_5x5_reduce=24, filters_5x5=64, filters_pool_proj=64,
                     name='inception_4c')

x = inception_module(x, filters_1x1=112, filters_3x3_reduce=144, filters_3x3=288,
                     filters_5x5_reduce=32, filters_5x5=64, filters_pool_proj=64,
                     name='inception_4d')

x = inception_module(x, filters_1x1=256, filters_3x3_reduce=160, filters_3x3=320,
                     filters_5x5_reduce=32, filters_5x5=128, filters_pool_proj=128,
                     name='inception_4e')

x = MaxPool2D((3,3), padding='same', strides=(1,1), name='max_pool_4_3x3_2')(x)

x = inception_module(x, filters_1x1=256, filters_3x3_reduce=160, filters_3x3=320,
                     filters_5x5_reduce=32, filters_5x5=128, filters_pool_proj=128, name='inception_5a')

x = inception_module(x, filters_1x1=384, filters_3x3_reduce=192, filters_3x3=384,
                     filters_5x5_reduce=48, filters_5x5=128, filters_pool_proj=128, name='inception_5b')

x = layers.GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
output = Dense(100, activation='softmax', name='output')(x)
model = models.Model(input, output)

In [ ]:
import math

epochs = 90
initial_lrate = 0.01

def decay(epoch, steps=100):
    initial_lrate = 0.01
    drop = 0.96
    epochs_drop = 8
    lrate = initial_lrate * math.pow(drop, math.floor((1+epoch)/epochs_drop))
    return lrate

lr_schedule = LearningRateScheduler(decay, verbose=1)
sgd = keras.optimizers.SGD(learning_rate=initial_lrate, momentum=0.9, nesterov=False)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])
model.fit(x_train, y_train, batch_size=256, epochs=epochs,
  validation_data=(x_test, y_test), callbacks=[lr_schedule], verbose=2, shuffle=True)

In [6]:
def bottleneck_residual_block(X, kernel_size, filters, reduce=False, s=2):
    F1, F2, F3 = filters

    X_shortcut = X

    if reduce:
        X_shortcut = Conv2D(filters = F3, kernel_size = (1, 1), strides =
        (s,s))(X_shortcut)
        X_shortcut = BatchNormalization(axis = 3)(X_shortcut)
        X = Conv2D(filters = F1, kernel_size = (1, 1), strides = (s,s), padding =
        'valid')(X)
        X = BatchNormalization(axis = 3)(X)
        X = layers.Activation('relu')(X)

    else:
        # First component of main path
        X = Conv2D(filters = F1, kernel_size = (1, 1), strides = (1,1), padding =
        'valid')(X)
        X = BatchNormalization(axis = 3)(X)
        X = layers.Activation('relu')(X)

    # Second component of main path
    X = Conv2D(filters = F2, kernel_size = kernel_size, strides = (1,1), padding =
    'same')(X)
    X = BatchNormalization(axis = 3)(X)
    X = layers.Activation('relu')(X)
    # Third component of main path
    X = Conv2D(filters = F3, kernel_size = (1, 1), strides = (1,1), padding =
    'valid')(X)
    X = BatchNormalization(axis = 3)(X)
    # Final step
    X = layers.Add()([X, X_shortcut])
    X = layers.Activation('relu')(X)
    return X

In [7]:
def ResNet40(input_shape, classes):
    X_input = layers.Input(input_shape)
    # Stage 1
    X = Conv2D(64, (7, 7), strides=(2, 2), name='conv1')(X_input)
    X = BatchNormalization(axis=3, name='bn_conv1')(X)
    X = layers.Activation('relu')(X)
    X = layers.MaxPooling2D((3, 3), strides=(2, 2))(X)
    # Stage 2
    X = bottleneck_residual_block(X, 3, [64, 64, 256], reduce=True, s=1)
    X = bottleneck_residual_block(X, 3, [64, 64, 256])
    X = bottleneck_residual_block(X, 3, [64, 64, 256])
    # Stage 3
    X = bottleneck_residual_block(X, 3, [128, 128, 512], reduce=True, s=2)
    X = bottleneck_residual_block(X, 3, [128, 128, 512])
    X = bottleneck_residual_block(X, 3, [128, 128, 512])
    X = bottleneck_residual_block(X, 3, [128, 128, 512])
    # Stage 4
    X = bottleneck_residual_block(X, 3, [256, 256, 1024], reduce=True, s=2)
    X = bottleneck_residual_block(X, 3, [256, 256, 1024])
    X = bottleneck_residual_block(X, 3, [256, 256, 1024])
    X = bottleneck_residual_block(X, 3, [256, 256, 1024])
    X = bottleneck_residual_block(X, 3, [256, 256, 1024])
    X = bottleneck_residual_block(X, 3, [256, 256, 1024])
    # AVGPOOL
    X = AveragePooling2D((1,1))(X)
    # output layer
    X = Flatten()(X)
    X = Dense(classes, activation='softmax', name='fc' + str(classes))(X)

    model = models.Model(inputs = X_input, outputs = X, name='ResNet50')
    return model

In [8]:
from keras.callbacks import ReduceLROnPlateau
model = ResNet40((32,32,3), 100)
epochs = 90
batch_size = 256
initial_lrate = 0.01
reduce_lr= ReduceLROnPlateau(monitor='val_loss',factor=np.sqrt(0.1),
    patience=5, min_lr=0.5e-6)
SGD = keras.optimizers.SGD(learning_rate=initial_lrate, momentum=0.9, nesterov=False)
model.compile(loss='categorical_crossentropy', optimizer=SGD, metrics=['accuracy'])
model.fit(x_train, y_train, batch_size=batch_size, validation_data=(x_test, y_test),
          epochs=epochs, callbacks=[reduce_lr]) 


Epoch 1/90
196/196 ━━━━━━━━━━━━━━━━━━━━ 174s 820ms/step - accuracy: 0.0699 - loss: 5.2984 - val_accuracy: 0.0531 - val_loss: 4.5117 - learning_rate: 0.0100
Epoch 2/90
196/196 ━━━━━━━━━━━━━━━━━━━━ 158s 808ms/step - accuracy: 0.1623 - loss: 3.6816 - val_accuracy: 0.1723 - val_loss: 3.5904 - learning_rate: 0.0100
Epoch 3/90
196/196 ━━━━━━━━━━━━━━━━━━━━ 159s 809ms/step - accuracy: 0.2147 - loss: 3.3538 - val_accuracy: 0.2179 - val_loss: 3.5217 - learning_rate: 0.0100
Epoch 4/90
196/196 ━━━━━━━━━━━━━━━━━━━━ 162s 828ms/step - accuracy: 0.2633 - loss: 3.0729 - val_accuracy: 0.2356 - val_loss: 3.3087 - learning_rate: 0.0100
Epoch 5/90
196/196 ━━━━━━━━━━━━━━━━━━━━ 3920s 20s/step - accuracy: 0.3138 - loss: 2.7954 - val_accuracy: 0.2587 - val_loss: 3.2002 - learning_rate: 0.0100
Epoch 6/90
196/196 ━━━━━━━━━━━━━━━━━━━━ 165s 841ms/step - accuracy: 0.3606 - loss: 2.5629 - val_accuracy: 0.2739 - val_loss: 3.0914 - learning_rate: 0.0100
Epoch 7/90
196/196 ━━━━━━━━━━━━━━━━━━━━ 165s 840ms/step - accurac

KeyboardInterrupt: 